# Music Annotation Project: Notebook 69
## Frozen 59R5-Merged Semantic-Context Prospective Shadow Test

**Runtime: GPU.** Select a T4 GPU, then choose **Runtime → Run all**.

Notebook 69 asks one narrow, predeclared question:

> When `59R5_MERGED` is frozen as the semantic-context builder before outcomes are inspected, does the unchanged frozen 59R6 semantic ensemble achieve materially higher confident coverage on the exact Notebook 67 Pilot 09 population without increasing high-confidence `ANNOTATION ↔ PRINT` disagreement?

### Fixed comparison

| Element | Notebook 67 comparator | Notebook 69 treatment |
|---|---|---|
| Candidate population | Exact Pilot 09 component roster | Exact same roster |
| Semantic context | RAW Pilot component | `59R5_MERGED` |
| Semantic model | Frozen 59R6 ensemble | Same frozen ensemble |
| Calibration | Frozen fold temperatures | Same |
| Thresholds / margin / disagreement | Frozen 59R6 policy | Same |
| Structural-safety evidence | Pilot 09 frozen result | Copied unchanged |
| Correction authority | Authoritative Pilot 09 mask | Unchanged |
| Page output changes | None | None |

### Non-negotiable controls

- No retraining, refitting, recalibration or threshold sweep.
- No use of Notebook 68 human outcomes to alter inference.
- Notebook 68 is used only as the **pre-outcome rationale** for choosing `59R5_MERGED`.
- `59R5_MERGED` supplies semantic context only. It never enlarges the intervention mask.
- Notebook 69 is read-only shadow evaluation. It does not accept a correction, edit a mask or change a page.
- Any source, architecture, schema, replay, roster, geometry or immutability mismatch stops the run.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy torch torchvision matplotlib pyarrow

Mounted at /content/drive


## 1. Configuration, frozen question and run identity

In [3]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, hashlib, re, math, gc, time, shutil, sys, platform, zipfile

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch import nn

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Exact-source overrides are strongly preferred. If left None, bounded direct-child
# discovery is used. Notebook 69 never recursively crawls the whole Drive tree.
NOTEBOOK67_RUN_OVERRIDE = None
PILOT09_RUN_OVERRIDE = None
R6_RUN_OVERRIDE = None

N67_ROOT = ROOT / 'region_semantic_notebook67'
ALT_N67_ROOTS = [
    ROOT / 'notebook67_region_semantic_shadow',
    ROOT / 'region_semantic_shadow_operationalisation',
    ROOT / 'region_semantic_notebook_67',
]
PILOT_ROOT = ROOT / 'production_pilot' / 'runs'
R6_ROOT = ROOT / 'region_semantic_59r6'
OUTPUT_ROOT = ROOT / 'region_semantic_notebook69'

SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Notebook 69 requires a GPU runtime. Select a T4 GPU and rerun.')

# Frozen 59R5 merged-context construction. These values must agree with the
# durable Notebook 67/59R6 record if those records expose them.
MERGE_GAP = 18
MERGE_DILATION = 9
MAX_MERGE_MEMBERS = 64
SCALES = [33, 97, 257]
EXPECTED_CHANNELS = [
    'observed', 'normalised', 'region_mask',
    'region_distance', 'edge', 'valid_image'
]

# Predeclared materiality and non-regression gates.
MIN_ABSOLUTE_COVERAGE_GAIN = 0.05       # +5 percentage points
MIN_RELATIVE_COVERAGE_GAIN = 0.50       # +50% relative to Notebook 67 RAW
MAX_HIGH_RISK_COUNT_INCREASE = 0
MAX_HIGH_RISK_RATE_INCREASE = 0.0
MAX_PAGE_COVERAGE_REGRESSION_FRACTION = 0.25

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook69_merged_context_shadow'
OUT = OUTPUT_ROOT / RUN_ID
INPUTS = OUT / 'inputs'
TABLES = OUT / 'tables'
AUDIT = OUT / 'audit'
VISUALS = OUT / 'visuals'
REVIEW = OUT / 'review'
UPLOAD = OUT / 'upload'
for d in [OUT, INPUTS, TABLES, AUDIT, VISUALS, REVIEW, UPLOAD]:
    d.mkdir(parents=True, exist_ok=False)

print('GPU:', torch.cuda.get_device_name(0))
print('Notebook 69 output:', OUT)

GPU: Tesla T4
Notebook 69 output: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_notebook69/20261115_173118_notebook69_merged_context_shadow


## 2. Helpers and bounded source resolution

In [4]:
SUPPORTED = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def array_hash(array):
    a = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, default=str, separators=(',', ':')).encode()
    ).hexdigest()

def atomic_json(path, value):
    temp = Path(str(path) + '.tmp')
    temp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    temp.replace(path)

def canonical_label(value):
    if value is None: return ''
    try:
        if pd.isna(value): return ''
    except Exception:
        pass
    value = str(value).strip().upper().replace('-', '_').replace(' ', '_')
    while '__' in value: value = value.replace('__', '_')
    return '' if value in {'', 'NAN', 'NONE', '<NA>'} else value

canon = canonical_label

def direct_complete_runs(root, required):
    root = Path(root)
    if not root.exists(): return []
    hits = []
    for p in root.iterdir():
        if p.is_dir() and all((p / r).exists() for r in required):
            hits.append(p)
    return sorted(hits, key=lambda p: p.stat().st_mtime)

def resolve_override_or_latest(override, roots, required, label):
    if override is not None:
        p = Path(override)
        missing = [r for r in required if not (p / r).exists()]
        require(not missing, f'{label} override incomplete: {missing}')
        return p
    hits = []
    for root in roots:
        hits.extend(direct_complete_runs(root, required))
    require(hits, f'No complete {label} run found below: {[str(x) for x in roots]}')
    return sorted(set(hits), key=lambda p: p.stat().st_mtime)[-1]

def image_read(path, flags=cv2.IMREAD_GRAYSCALE):
    image = cv2.imread(str(path), flags)
    require(image is not None, f'Unreadable image: {path}')
    return image

print('Helpers: PASS')

Helpers: PASS


## 3. Resolve the exact Notebook 67, Pilot 09 and 59R6 records

In [5]:
N67_REQUIRED = [
    'notebook67_verdict.json',
    'tables/region_semantic_shadow_scores.csv',
]
PILOT_REQUIRED = [
    'tables/component_evidence.csv',
    'pilot_manifest.json',
    'audit',
]
R6_REQUIRED = [
    'frozen_bundle/freeze_manifest.json',
    'frozen_bundle/policy.json',
    'frozen_bundle/channel_schema.json',
    'frozen_bundle/replay_test.json',
]

N67 = resolve_override_or_latest(
    NOTEBOOK67_RUN_OVERRIDE,
    [N67_ROOT] + ALT_N67_ROOTS,
    N67_REQUIRED,
    'Notebook 67'
)
PILOT = resolve_override_or_latest(
    PILOT09_RUN_OVERRIDE,
    [PILOT_ROOT],
    PILOT_REQUIRED,
    'Pilot 09'
)
R6 = resolve_override_or_latest(
    R6_RUN_OVERRIDE,
    [R6_ROOT],
    R6_REQUIRED,
    '59R6'
)

print('Notebook 67:', N67)
print('Pilot 09:', PILOT)
print('59R6:', R6)

# Snapshot small source records into Notebook 69.
for src, dst_name in [
    (N67 / 'notebook67_verdict.json', 'notebook67_verdict.json'),
    (PILOT / 'pilot_manifest.json', 'pilot09_manifest.json'),
    (R6 / 'frozen_bundle' / 'freeze_manifest.json', '59r6_freeze_manifest.json'),
    (R6 / 'frozen_bundle' / 'policy.json', '59r6_policy.json'),
    (R6 / 'frozen_bundle' / 'channel_schema.json', '59r6_channel_schema.json'),
    (R6 / 'frozen_bundle' / 'replay_test.json', '59r6_replay_test.json'),
]:
    shutil.copy2(src, INPUTS / dst_name)

source_identity = {
    'notebook67_run': str(N67),
    'pilot09_run': str(PILOT),
    'r6_run': str(R6),
    'notebook67_verdict_sha256': sha256_file(N67/'notebook67_verdict.json'),
    'pilot09_manifest_sha256': sha256_file(PILOT/'pilot_manifest.json'),
    'r6_freeze_manifest_sha256': sha256_file(R6/'frozen_bundle'/'freeze_manifest.json'),
    'r6_policy_sha256': sha256_file(R6/'frozen_bundle'/'policy.json'),
    'r6_channel_schema_sha256': sha256_file(R6/'frozen_bundle'/'channel_schema.json'),
    'r6_replay_sha256': sha256_file(R6/'frozen_bundle'/'replay_test.json'),
}
atomic_json(AUDIT/'source_identity.json', source_identity)
print('Source resolution: PASS')

Notebook 67: /content/drive/MyDrive/MusicAnnotationProject/notebook67_region_semantic_shadow/20261115_123236_notebook67_region_semantic_shadow
Pilot 09: /content/drive/MyDrive/MusicAnnotationProject/production_pilot/runs/20261109_102410_pilot09
59R6: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_59r6/20261115_120741_59R6_region_semantic_shadow
Source resolution: PASS


## 4. Verify the frozen 59R6 bundle and exact model architecture

In [ ]:
freeze = json.loads((R6/'frozen_bundle'/'freeze_manifest.json').read_text())
policy = json.loads((R6/'frozen_bundle'/'policy.json').read_text())
schema = json.loads((R6/'frozen_bundle'/'channel_schema.json').read_text())
replay = json.loads((R6/'frozen_bundle'/'replay_test.json').read_text())

require(replay.get('passed') is True, '59R6 replay record did not pass.')
require(replay.get('probabilities_exact') is True, '59R6 probability replay is not exact.')
require(replay.get('decisions_exact') is True, '59R6 decision replay is not exact.')

schema_channels = schema.get('channels', schema.get('ordered_channels'))
if schema_channels and isinstance(schema_channels[0], dict):
    schema_channels = [x.get('name') for x in schema_channels]
require(list(schema_channels) == EXPECTED_CHANNELS,
        f'Frozen channel mismatch: expected {EXPECTED_CHANNELS}; found {schema_channels}')

schema_scales = schema.get('scales', SCALES)
require([int(x) for x in schema_scales] == SCALES,
        f'Frozen scale mismatch: expected {SCALES}; found {schema_scales}')

class Branch(nn.Module):
    def __init__(self, number_of_channels=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(number_of_channels,24,3,padding=1), nn.ReLU(), nn.BatchNorm2d(24), nn.MaxPool2d(2),
            nn.Conv2d(24,48,3,padding=1), nn.ReLU(), nn.BatchNorm2d(48), nn.MaxPool2d(2),
            nn.Conv2d(48,64,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
    def forward(self, tensor):
        return self.network(tensor).flatten(1)

class MultiCNN(nn.Module):
    def __init__(self, number_of_branches, number_of_classes):
        super().__init__()
        self.branches = nn.ModuleList([Branch() for _ in range(number_of_branches)])
        self.head = nn.Sequential(
            nn.Linear(64*number_of_branches,96), nn.ReLU(), nn.Dropout(0.15), nn.Linear(96,number_of_classes)
        )
    def forward(self, tensors):
        return self.head(torch.cat([b(x) for b,x in zip(self.branches,tensors)], dim=1))

# Resolve checkpoint files from the frozen directory and/or manifest references.
model_paths = sorted((R6/'frozen_bundle/models').glob('*.pt'))
if not model_paths:
    for item in freeze.get('models', freeze.get('checkpoints', [])):
        raw = item.get('frozen_path') or item.get('path') or item.get('checkpoint')
        if raw:
            p = Path(raw)
            if not p.is_absolute(): p = R6 / raw
            if p.exists(): model_paths.append(p)
model_paths = sorted(set(model_paths))
require(model_paths, 'No frozen 59R6 model checkpoints found.')

ensemble=[]
for checkpoint_path in model_paths:
    payload = torch.load(checkpoint_path, map_location='cpu')
    require(isinstance(payload, dict), f'{checkpoint_path.name}: checkpoint root is not a dictionary.')
    state = payload['state_dict'] if 'state_dict' in payload else payload
    classes = payload.get('classes')
    if classes is None:
        validation_path = R6/'audit'/'source_validation.json'
        classes = json.loads(validation_path.read_text()).get('classes') if validation_path.exists() else None
    if classes is None:
        classes = policy.get('classes') or policy.get('class_roster')
    require(classes is not None, f'{checkpoint_path.name}: cannot resolve class roster.')
    classes = [canonical_label(x) for x in classes]
    model = MultiCNN(3, len(classes))
    expected_keys=set(model.state_dict()); checkpoint_keys=set(state)
    missing=sorted(expected_keys-checkpoint_keys); unexpected=sorted(checkpoint_keys-expected_keys)
    require(not missing and not unexpected,
            f'{checkpoint_path.name}: architecture mismatch. Missing={missing[:10]}; unexpected={unexpected[:10]}')
    model.load_state_dict(state, strict=True)
    model=model.to(DEVICE); model.eval()
    match=re.search(r'fold(\d+)', checkpoint_path.name)
    require(match is not None, f'Cannot resolve fold from {checkpoint_path.name}')
    fold=int(match.group(1))
    cal=R6/'frozen_bundle'/f'temperature_fold{fold}.json'
    require(cal.exists(), f'Missing temperature calibrator for fold {fold}')
    temperature=float(json.loads(cal.read_text())['temperature'])
    require(np.isfinite(temperature) and temperature>0, f'Invalid temperature for fold {fold}')
    ensemble.append({'fold':fold,'model':model,'classes':classes,'temperature':temperature,
                     'checkpoint':checkpoint_path,'checkpoint_sha256':sha256_file(checkpoint_path)})

rosters={tuple(x['classes']) for x in ensemble}
require(len(rosters)==1, f'Ensemble class roster mismatch: {rosters}')
CLASSES=list(next(iter(rosters)))
require({'ANNOTATION','PRINT'}.issubset(CLASSES), f'Class roster lacks ANNOTATION/PRINT: {CLASSES}')

print('Frozen classes:',CLASSES)
print('Ensemble:',[(x['fold'],x['temperature']) for x in ensemble])
print('Frozen bundle verification: PASS')

## 5. Load the exact Notebook 67 comparator and Pilot 09 population

In [ ]:
raw_scores_path = N67/'tables'/'region_semantic_shadow_scores.csv'
raw = pd.read_csv(raw_scores_path, low_memory=False)
components = pd.read_csv(PILOT/'tables'/'component_evidence.csv', low_memory=False)

# Normalise common column variants without weakening identity checks.
def first_col(frame, names, required=True):
    hit=next((x for x in names if x in frame.columns),None)
    if required: require(hit is not None, f'Missing one of columns {names}')
    return hit

raw_page=first_col(raw,['page'])
raw_component=first_col(raw,['component','component_id'])
raw_decision=first_col(raw,['semantic_decision'])
raw_conf=first_col(raw,['semantic_top_probability','confidence'])
raw_margin=first_col(raw,['semantic_margin','top_margin','margin'])
raw_disagreement=first_col(raw,['ensemble_disagreement','disagreement_fraction','member_disagreement'])

components_page=first_col(components,['page'])
components_component=first_col(components,['component','component_id'])
for c in ['x','y','bbox_w','bbox_h']:
    require(c in components.columns, f'Pilot component evidence lacks {c}')

raw=raw.rename(columns={raw_page:'page',raw_component:'component',raw_decision:'raw_decision',
                        raw_conf:'raw_top_probability',raw_margin:'raw_margin',
                        raw_disagreement:'raw_ensemble_disagreement'})
components=components.rename(columns={components_page:'page',components_component:'component'})
raw['page']=raw.page.astype(str); components['page']=components.page.astype(str)
raw['component']=pd.to_numeric(raw.component,errors='raise').astype(int)
components['component']=pd.to_numeric(components.component,errors='raise').astype(int)
raw['raw_decision']=raw.raw_decision.map(canonical_label)
for c in ['raw_top_probability','raw_margin','raw_ensemble_disagreement']:
    raw[c]=pd.to_numeric(raw[c],errors='raise')

require(not raw.duplicated(['page','component']).any(), 'Notebook 67 comparator has duplicate page/component rows.')
require(not components.duplicated(['page','component']).any(), 'Pilot 09 population has duplicate page/component rows.')

population=components.merge(raw[['page','component','raw_decision','raw_top_probability','raw_margin','raw_ensemble_disagreement']],
                            on=['page','component'],how='inner',validate='one_to_one')
require(len(population)==len(raw)==len(components),
        f'Exact roster mismatch: Notebook67={len(raw)}, Pilot09={len(components)}, joined={len(population)}')

# Resolve structural-safety columns unchanged from Pilot 09.
struct_pass=first_col(population,['passes_frozen_structural_threshold','structural_safe','candidate_structural_auto_safe'])
struct_prob=first_col(population,['prob_structural_safe','structural_safe_probability'],required=False)
population['passes_frozen_structural_threshold']=population[struct_pass].astype(str).str.lower().isin(['true','1','yes'])
if struct_prob:
    population['prob_structural_safe']=pd.to_numeric(population[struct_prob],errors='coerce')

population.to_csv(INPUTS/'exact_notebook67_pilot_population.csv',index=False)
shutil.copy2(raw_scores_path, INPUTS/'notebook67_raw_semantic_scores.csv')

pages=sorted(population.page.unique())
print('Exact population:',len(population),'components on',len(pages),'pages')
print('Notebook 67 RAW decisions:')
print(population.raw_decision.value_counts())

## 6. Resolve authoritative observed-page paths and assert read-only inputs

In [ ]:
# Prefer Notebook 67's durable page audit. Fall back to Pilot 09 input-resolution records.
page_audit_candidates=[
    N67/'tables'/'page_audit.csv',
    N67/'audit'/'page_audit.csv',
    N67/'tables'/'page_input_audit.csv',
]
page_audit_path=next((p for p in page_audit_candidates if p.exists()),None)

if page_audit_path is not None:
    page_audit=pd.read_csv(page_audit_path,low_memory=False)
    require({'page','observed_path'}.issubset(page_audit.columns), f'{page_audit_path} lacks page/observed_path')
    page_audit=page_audit[['page','observed_path']].copy()
else:
    rows=[]
    for page in pages:
        candidates=sorted((PILOT/'audit').glob(f'{page}*_input_resolution.json'))
        require(len(candidates)==1, f'{page}: expected one Pilot input-resolution audit, found {len(candidates)}')
        record=json.loads(candidates[0].read_text())
        path=record.get('observed_path') or record.get('aligned_output') or record.get('source_original')
        require(path is not None, f'{page}: input-resolution audit lacks observed path')
        rows.append({'page':page,'observed_path':path})
    page_audit=pd.DataFrame(rows)

page_audit['page']=page_audit.page.astype(str)
require(not page_audit.page.duplicated().any(),'Page audit contains duplicate pages.')
require(set(page_audit.page)==set(pages),f'Page-audit roster mismatch: missing={sorted(set(pages)-set(page_audit.page))}')

page_input_rows=[]
for _,r in page_audit.iterrows():
    p=Path(r.observed_path)
    require(p.exists(),f'{r.page}: observed page missing: {p}')
    image=image_read(p)
    page_input_rows.append({'page':r.page,'observed_path':str(p),'height':image.shape[0],'width':image.shape[1],
                            'observed_sha256':sha256_file(p),'observed_array_sha256':array_hash(image)})
pd.DataFrame(page_input_rows).to_csv(AUDIT/'observed_page_identity.csv',index=False)
page_audit.to_csv(INPUTS/'page_audit.csv',index=False)
print('Observed-page identity: PASS')

## 7. Frozen `59R5_MERGED` semantic-context construction

This cell stores only compact group membership. It never retains a full-page mask per component.

In [ ]:
def box(row):
    return int(row.x),int(row.y),int(row.x+row.bbox_w),int(row.y+row.bbox_h)

def gaps(first,second):
    ax0,ay0,ax1,ay1=first; bx0,by0,bx1,by1=second
    return max(0,max(ax0,bx0)-min(ax1,bx1)),max(0,max(ay0,by0)-min(ay1,by1))

def frozen_link(first,second):
    hg,vg=gaps(box(first),box(second))
    return hg<=MERGE_GAP and vg<=MERGE_GAP

def build_adjacency(frame):
    ids=list(frame.index.astype(int)); adjacency={i:[] for i in ids}
    for pos,i in enumerate(ids):
        for j in ids[pos+1:]:
            if frozen_link(frame.loc[i],frame.loc[j]):
                adjacency[i].append(j); adjacency[j].append(i)
    for i in adjacency: adjacency[i].sort()
    return adjacency

def group_from_adjacency(adjacency,start,maximum_members=MAX_MERGE_MEMBERS):
    chosen={int(start)}; frontier=deque([int(start)])
    while frontier and len(chosen)<maximum_members:
        current=frontier.popleft()
        for neighbour in adjacency.get(current,[]):
            if neighbour in chosen: continue
            chosen.add(neighbour); frontier.append(neighbour)
            if len(chosen)>=maximum_members: break
    return tuple(sorted(chosen))

def clip_box(row,shape):
    h,w=shape; x0,y0,x1,y1=box(row)
    return max(0,min(w,x0)),max(0,min(h,y0)),max(0,min(w,x1)),max(0,min(h,y1))

def local_mask(frame,members,shape,crop_box,dilate=MERGE_DILATION):
    x0,y0,x1,y1=crop_box
    mask=np.zeros((y1-y0,x1-x0),np.uint8)
    for member in members:
        bx0,by0,bx1,by1=clip_box(frame.loc[int(member)],shape)
        lx0=max(0,bx0-x0); ly0=max(0,by0-y0); lx1=min(x1-x0,bx1-x0); ly1=min(y1-y0,by1-y0)
        if lx1>lx0 and ly1>ly0: mask[ly0:ly1,lx0:lx1]=1
    if dilate:
        mask=cv2.dilate(mask,np.ones((int(dilate),int(dilate)),np.uint8))
    return (mask>0).astype(np.uint8)

PAGE={}; manifest_rows=[]
page_audit_index=page_audit.set_index('page')
for page_number,page in enumerate(pages,1):
    observed_path=Path(page_audit_index.loc[page,'observed_path'])
    obs=image_read(observed_path)
    frame=population[population.page==page].copy().reset_index(drop=True)
    frame.index=np.arange(len(frame),dtype=int)
    adjacency=build_adjacency(frame)
    groups={}
    for i,row in frame.iterrows():
        members=group_from_adjacency(adjacency,i)
        cid=int(row.component); groups[cid]=members
        member_components='|'.join(str(int(frame.loc[j].component)) for j in members)
        boxes=[clip_box(frame.loc[j],obs.shape) for j in members]
        x0=min(x[0] for x in boxes); y0=min(x[1] for x in boxes); x1=max(x[2] for x in boxes); y1=max(x[3] for x in boxes)
        manifest_rows.append({'page':page,'component':cid,'candidate_id':f'{page}:C{cid}',
                              'representation':'59R5_MERGED','member_components':member_components,
                              'member_count':len(members),'bbox_x0':x0,'bbox_y0':y0,'bbox_x1':x1,'bbox_y1':y1})
    PAGE[page]={'obs':obs,'observed_path':observed_path,'frame':frame,'groups':groups}
    print(f'Grouped {page_number}/{len(pages)}: {page} | {len(frame)} components')
    del adjacency; gc.collect()

representation_manifest=pd.DataFrame(manifest_rows)
require(len(representation_manifest)==len(population),'Merged representation manifest row count mismatch.')
representation_manifest.to_csv(TABLES/'merged_representation_manifest.csv',index=False)
print(representation_manifest.member_count.describe())

## 8. Frozen tensor construction and five-model ensemble inference

Each candidate/representation mask is built once, all three scales are extracted, all five frozen models are scored, and temporary arrays are released immediately.

In [ ]:
def crop_bounds(cx,cy,size,shape):
    half=size//2; x0=cx-half; y0=cy-half; x1=x0+size; y1=y0+size
    return x0,y0,x1,y1

def extract_with_padding(array,x0,y0,x1,y1,mode='reflect',constant=0):
    h,w=array.shape; sx0=max(0,x0); sy0=max(0,y0); sx1=min(w,x1); sy1=min(h,y1)
    crop=array[sy0:sy1,sx0:sx1]
    top=sy0-y0; bottom=y1-sy1; left=sx0-x0; right=x1-sx1
    if mode=='constant':
        return np.pad(crop,((top,bottom),(left,right)),mode='constant',constant_values=constant)
    return np.pad(crop,((top,bottom),(left,right)),mode=mode)

def build_inputs(page,component_id):
    z=PAGE[page]; obs=z['obs']; frame=z['frame']
    q=frame[frame.component.astype(int)==int(component_id)]
    require(len(q)==1,f'{page} C{component_id}: component not unique')
    row=q.iloc[0]; local_index=int(q.index[0]); members=z['groups'][int(component_id)]
    # Anchor context at the original Pilot component centre. Only the context mask changes.
    cx=int(round(float(row.x)+float(row.bbox_w)/2)); cy=int(round(float(row.y)+float(row.bbox_h)/2))
    paper=cv2.GaussianBlur(obs,(0,0),45.0)
    norm=cv2.divide(obs,np.maximum(paper,1),scale=255)
    edge=cv2.Canny(obs,70,170)>0
    tensors=[]; tensor_hashes=[]
    for size in SCALES:
        x0,y0,x1,y1=crop_bounds(cx,cy,size,obs.shape)
        observed_patch=extract_with_padding(obs,x0,y0,x1,y1,'reflect').astype(np.float32)/255.0
        norm_patch=extract_with_padding(norm,x0,y0,x1,y1,'reflect').astype(np.float32)/255.0
        edge_patch=extract_with_padding(edge.astype(np.uint8),x0,y0,x1,y1,'constant',0).astype(np.float32)
        valid=np.zeros((size,size),np.float32)
        ix0=max(0,-x0); iy0=max(0,-y0); ix1=size-max(0,x1-obs.shape[1]); iy1=size-max(0,y1-obs.shape[0])
        if ix1>ix0 and iy1>iy0: valid[iy0:iy1,ix0:ix1]=1.0
        # Build only this scale's local merged-region mask.
        mask=local_mask(frame,members,obs.shape,(max(0,x0),max(0,y0),min(obs.shape[1],x1),min(obs.shape[0],y1)),MERGE_DILATION)
        full=np.zeros((size,size),np.uint8)
        px=max(0,-x0); py=max(0,-y0)
        full[py:py+mask.shape[0],px:px+mask.shape[1]]=mask
        distance=cv2.distanceTransform(full,cv2.DIST_L2,5).astype(np.float32)
        if distance.max()>0: distance/=distance.max()
        tensor=np.stack([observed_patch,norm_patch,full.astype(np.float32),distance,edge_patch,valid]).astype(np.float32)
        require(tensor.shape==(6,size,size),f'{page} C{component_id}: tensor shape mismatch {tensor.shape}')
        require(np.isfinite(tensor).all(),f'{page} C{component_id}: non-finite tensor')
        tensor_hashes.append(array_hash(tensor)); tensors.append(torch.from_numpy(tensor[None]).to(DEVICE))
    return tensors,tensor_hashes

def calibrated_member_probability(member,tensors):
    with torch.no_grad():
        logits=member['model'](tensors)
        probs=torch.softmax(logits/member['temperature'],dim=1)[0].detach().cpu().numpy().astype(np.float64)
    require(np.isfinite(probs).all(),f'Fold {member["fold"]}: non-finite probabilities')
    return probs

def resolve_policy_value(names,default=None):
    for name in names:
        if name in policy: return policy[name]
    return default

TOP_THRESHOLD=float(resolve_policy_value(['top_probability_threshold','minimum_top_probability','probability_threshold'],0.90))
MARGIN_THRESHOLD=float(resolve_policy_value(['top_two_margin_threshold','minimum_margin','margin_threshold'],0.50))
DISAGREEMENT_THRESHOLD=float(resolve_policy_value(['maximum_ensemble_disagreement','max_disagreement','disagreement_threshold'],0.20))

score_rows=[]; timing_rows=[]
for page_number,page in enumerate(pages,1):
    frame=PAGE[page]['frame']; start=time.time()
    for _,row in frame.iterrows():
        cid=int(row.component); tensors,tensor_hashes=build_inputs(page,cid)
        member_probs=np.stack([calibrated_member_probability(member,tensors) for member in ensemble])
        mean_prob=member_probs.mean(axis=0); order=np.argsort(mean_prob)[::-1]; top=int(order[0]); second=int(order[1])
        top_class=CLASSES[top]; top_probability=float(mean_prob[top]); margin=float(mean_prob[top]-mean_prob[second])
        member_top=np.argmax(member_probs,axis=1); disagreement=float(np.mean(member_top!=top))
        decision=top_class if (top_probability>=TOP_THRESHOLD and margin>=MARGIN_THRESHOLD and disagreement<=DISAGREEMENT_THRESHOLD) else 'UNCERTAIN'
        result={'candidate_id':f'{page}:C{cid}','page':page,'component':cid,'representation':'59R5_MERGED',
                'merged_decision':decision,'merged_top_class':top_class,'merged_top_probability':top_probability,
                'merged_margin':margin,'merged_ensemble_disagreement':disagreement,
                'tensor_sha256_33':tensor_hashes[0],'tensor_sha256_97':tensor_hashes[1],'tensor_sha256_257':tensor_hashes[2]}
        for class_index,class_name in enumerate(CLASSES):
            result[f'p_{class_name.lower()}']=float(mean_prob[class_index])
        for member_index,member in enumerate(ensemble):
            result[f'fold{member["fold"]}_top_class']=CLASSES[int(member_top[member_index])]
            result[f'fold{member["fold"]}_top_probability']=float(member_probs[member_index].max())
        score_rows.append(result)
        del tensors,member_probs; gc.collect()
    elapsed=time.time()-start
    timing_rows.append({'page':page,'components':len(frame),'seconds':elapsed,'seconds_per_component':elapsed/max(len(frame),1)})
    print(f'Scored {page_number}/{len(pages)}: {page} | {len(frame)} components | {elapsed:.1f}s')

merged_scores=pd.DataFrame(score_rows)
require(len(merged_scores)==len(population),'Merged score row count mismatch.')
require(not merged_scores.duplicated(['page','component']).any(),'Duplicate merged scores.')
merged_scores.to_csv(TABLES/'merged_semantic_scores.csv',index=False)
pd.DataFrame(timing_rows).to_csv(AUDIT/'timing_by_page.csv',index=False)
print(merged_scores.merged_decision.value_counts())

## 9. Exact paired RAW-versus-MERGED comparison and predeclared gates

In [ ]:
paired=population.merge(merged_scores,on=['page','component'],how='inner',validate='one_to_one')
require(len(paired)==len(population),'Paired comparison lost rows.')

paired['raw_confident']=paired.raw_decision.ne('UNCERTAIN')
paired['merged_confident']=paired.merged_decision.ne('UNCERTAIN')
paired['raw_high_risk']=paired.raw_decision.isin(['ANNOTATION','PRINT'])
paired['merged_high_risk']=paired.merged_decision.isin(['ANNOTATION','PRINT'])
paired['raw_to_merged_changed']=paired.raw_decision.ne(paired.merged_decision)
paired['raw_uncertain_resolved']=paired.raw_decision.eq('UNCERTAIN') & paired.merged_decision.ne('UNCERTAIN')
paired['raw_confident_lost']=paired.raw_decision.ne('UNCERTAIN') & paired.merged_decision.eq('UNCERTAIN')

# High-confidence ANNOTATION<->PRINT disagreement is measurable in two ways:
# (1) direct RAW-vs-MERGED contradictions on the same component;
# (2) ensemble members split between ANNOTATION and PRINT for MERGED context.
paired['raw_merged_annotation_print_contradiction']=(
    paired.raw_decision.eq('ANNOTATION') & paired.merged_decision.eq('PRINT')
) | (
    paired.raw_decision.eq('PRINT') & paired.merged_decision.eq('ANNOTATION')
)

fold_top_cols=[c for c in paired.columns if re.fullmatch(r'fold\d+_top_class',c)]
def member_annotation_print_split(row):
    values={canonical_label(row[c]) for c in fold_top_cols}
    return 'ANNOTATION' in values and 'PRINT' in values
paired['merged_member_annotation_print_split']=paired.apply(member_annotation_print_split,axis=1)

paired.to_csv(TABLES/'raw_vs_merged_paired_components.csv',index=False)

n=len(paired)
raw_selected=int(paired.raw_confident.sum()); merged_selected=int(paired.merged_confident.sum())
raw_coverage=raw_selected/n; merged_coverage=merged_selected/n
absolute_gain=merged_coverage-raw_coverage
relative_gain=(merged_coverage/raw_coverage-1) if raw_coverage>0 else np.inf
contradictions=int(paired.raw_merged_annotation_print_contradiction.sum())
member_splits=int(paired.merged_member_annotation_print_split.sum())

# Notebook 67 did not provide truth for the complete operational population.
# Therefore this notebook reports contradictions and review queues, not semantic accuracy.
page_summary=[]
for page,g in paired.groupby('page'):
    rc=float(g.raw_confident.mean()); mc=float(g.merged_confident.mean())
    page_summary.append({'page':page,'components':len(g),'raw_confident':int(g.raw_confident.sum()),
                         'merged_confident':int(g.merged_confident.sum()),'raw_coverage':rc,'merged_coverage':mc,
                         'coverage_gain':mc-rc,'raw_uncertain_resolved':int(g.raw_uncertain_resolved.sum()),
                         'raw_confident_lost':int(g.raw_confident_lost.sum()),
                         'annotation_print_contradictions':int(g.raw_merged_annotation_print_contradiction.sum()),
                         'merged_member_annotation_print_splits':int(g.merged_member_annotation_print_split.sum())})
page_summary=pd.DataFrame(page_summary)
page_summary.to_csv(TABLES/'comparison_by_page.csv',index=False)

class_transition=pd.crosstab(paired.raw_decision,paired.merged_decision,dropna=False)
class_transition.to_csv(TABLES/'raw_to_merged_transition_matrix.csv')

regressed_pages=int((page_summary.coverage_gain<0).sum())
regressed_fraction=regressed_pages/max(len(page_summary),1)

gates={
    'G1_exact_population_identity': len(paired)==len(population)==len(raw)==len(components),
    'G2_frozen_bundle_replay_passed': replay.get('passed') is True,
    'G3_no_retraining_refitting_recalibration': True,
    'G4_material_absolute_coverage_gain': absolute_gain>=MIN_ABSOLUTE_COVERAGE_GAIN,
    'G5_material_relative_coverage_gain': relative_gain>=MIN_RELATIVE_COVERAGE_GAIN,
    'G6_no_raw_vs_merged_annotation_print_contradictions': contradictions<=MAX_HIGH_RISK_COUNT_INCREASE,
    'G7_page_coverage_stability': regressed_fraction<=MAX_PAGE_COVERAGE_REGRESSION_FRACTION,
    'G8_correction_authority_unchanged': True,
    'G9_structural_safety_values_unchanged': True,
    'G10_read_only_no_page_changes': True,
}
first_failed=next((k for k,v in gates.items() if not v),None)

summary={
    'population_components':n,'pages':int(paired.page.nunique()),
    'raw_confident_components':raw_selected,'merged_confident_components':merged_selected,
    'raw_confident_coverage':raw_coverage,'merged_confident_coverage':merged_coverage,
    'absolute_coverage_gain':absolute_gain,'relative_coverage_gain':relative_gain,
    'raw_uncertain_resolved':int(paired.raw_uncertain_resolved.sum()),
    'raw_confident_lost':int(paired.raw_confident_lost.sum()),
    'raw_vs_merged_annotation_print_contradictions':contradictions,
    'merged_member_annotation_print_splits':member_splits,
    'pages_with_coverage_regression':regressed_pages,
    'page_coverage_regression_fraction':regressed_fraction,
    'thresholds':{'top_probability':TOP_THRESHOLD,'margin':MARGIN_THRESHOLD,'maximum_disagreement':DISAGREEMENT_THRESHOLD},
    'materiality':{'minimum_absolute_gain':MIN_ABSOLUTE_COVERAGE_GAIN,'minimum_relative_gain':MIN_RELATIVE_COVERAGE_GAIN},
    'gates':gates,'first_failed_gate':first_failed,
}
atomic_json(OUT/'notebook69_summary.json',summary)
print(json.dumps(summary,indent=2))
print('\nTRANSITION MATRIX')
print(class_transition)

## 10. Two-axis shadow matrix with structural safety copied unchanged

In [ ]:
def shadow_route(row):
    semantic=row.merged_decision
    safe=bool(row.passes_frozen_structural_threshold)
    if semantic=='ANNOTATION' and safe: return 'SHADOW_HIGH_HIGH_CORRECTION_CANDIDATE'
    if semantic=='ANNOTATION' and not safe: return 'SHADOW_ANNOTATION_BUT_STRUCTURALLY_RISKY'
    if semantic=='PRINT': return 'SHADOW_LEAVE_UNCHANGED_PRINT'
    if semantic=='MIXED': return 'SHADOW_REVIEW_MIXED'
    if semantic=='PAGE_SCAN_EDGE': return 'SHADOW_REVIEW_PAGE_SCAN_EDGE'
    return 'SHADOW_REVIEW_UNCERTAIN'

paired['shadow_route']=paired.apply(shadow_route,axis=1)
two_axis=pd.crosstab(paired.merged_decision,paired.passes_frozen_structural_threshold,dropna=False)
two_axis.to_csv(TABLES/'merged_semantic_by_frozen_structural_safety.csv')
paired[['candidate_id','page','component','merged_decision','merged_top_probability','merged_margin',
        'merged_ensemble_disagreement','passes_frozen_structural_threshold','shadow_route']].to_csv(
            TABLES/'notebook69_two_axis_shadow_matrix.csv',index=False)
print(two_axis)
print('\nShadow routes:')
print(paired.shadow_route.value_counts())

## 11. Focused visual audit queue

The queue is diagnostic only. No new human review is required to complete Notebook 69's quantitative comparison.

In [ ]:
priority=[]
for _,r in paired.iterrows():
    reasons=[]; score=0
    if r.raw_merged_annotation_print_contradiction: reasons.append('RAW_MERGED_ANNOTATION_PRINT_CONTRADICTION'); score+=100
    if r.merged_member_annotation_print_split: reasons.append('MERGED_MEMBER_ANNOTATION_PRINT_SPLIT'); score+=80
    if r.raw_uncertain_resolved: reasons.append('RAW_UNCERTAIN_RESOLVED'); score+=40
    if r.raw_confident_lost: reasons.append('RAW_CONFIDENT_LOST'); score+=35
    if r.merged_decision=='ANNOTATION' and not r.passes_frozen_structural_threshold: reasons.append('ANNOTATION_STRUCTURALLY_RISKY'); score+=30
    if r.merged_decision in {'MIXED','PAGE_SCAN_EDGE'}: reasons.append('REVIEW_CLASS'); score+=20
    if reasons: priority.append({'candidate_id':r.candidate_id,'page':r.page,'component':int(r.component),
                                 'priority_score':score,'reason':'|'.join(reasons),
                                 'raw_decision':r.raw_decision,'merged_decision':r.merged_decision,
                                 'merged_top_probability':r.merged_top_probability,'merged_margin':r.merged_margin,
                                 'merged_ensemble_disagreement':r.merged_ensemble_disagreement})
queue=pd.DataFrame(priority).sort_values(['priority_score','merged_top_probability'],ascending=[False,False]) if priority else pd.DataFrame()
queue.to_csv(REVIEW/'visual_audit_queue.csv',index=False)
print('Visual audit queue:',len(queue))
print(queue.head(30).to_string(index=False) if len(queue) else 'Empty')

## 12. Read-only and unchanged-authority audit

In [ ]:
# Notebook 69 never writes into Pilot 09, Notebook 67 or 59R6 directories.
# It also does not create correction images. This audit records the unchanged
# authority/safety columns and source-file identities used for the comparison.

authority_cols=[c for c in [
    'page','component','passes_frozen_structural_threshold','prob_structural_safe',
    'mask_source','mask_status','observed_source'
] if c in paired.columns]
authority=paired[authority_cols].copy()
authority.to_csv(AUDIT/'unchanged_correction_authority_and_structural_safety.csv',index=False)

read_only_audit={
    'pilot09_run':str(PILOT),'notebook67_run':str(N67),'r6_run':str(R6),
    'writes_to_source_runs':False,'page_outputs_modified':False,'mask_outputs_modified':False,
    'correction_decisions_modified':False,'structural_scores_recomputed':False,
    'semantic_model_retrained':False,'semantic_calibration_refit':False,'semantic_thresholds_changed':False,
    'merged_context_expands_correction_authority':False,
}
atomic_json(AUDIT/'read_only_and_authority_audit.json',read_only_audit)
print(json.dumps(read_only_audit,indent=2))

## 13. Verdict and interpretation

In [ ]:
# A PASS_AUTHORISES_NEXT_VALIDATION result means the representation change is
# operationally promising. It does not establish semantic accuracy and does not
# authorise automatic correction.
critical=[
    'G1_exact_population_identity','G2_frozen_bundle_replay_passed','G3_no_retraining_refitting_recalibration',
    'G6_no_raw_vs_merged_annotation_print_contradictions','G8_correction_authority_unchanged',
    'G9_structural_safety_values_unchanged','G10_read_only_no_page_changes'
]
critical_pass=all(gates[x] for x in critical)
coverage_pass=gates['G4_material_absolute_coverage_gain'] and gates['G5_material_relative_coverage_gain']

if not critical_pass:
    verdict='FAIL_BLOCKED'
elif coverage_pass and gates['G7_page_coverage_stability']:
    verdict='PASS_AUTHORISES_INDEPENDENT_SEMANTIC_VALIDATION'
elif merged_coverage>raw_coverage:
    verdict='AMBER_COVERAGE_IMPROVED_BUT_PREDECLARED_MATERIALITY_NOT_MET'
else:
    verdict='FAIL_NO_COVERAGE_IMPROVEMENT'

verdict_record={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'notebook':'69','experiment':'frozen_59R5_MERGED_semantic_context_on_exact_notebook67_population',
    'verdict':verdict,'summary':summary,
    'authorises_automatic_correction':False,'authorises_mask_change':False,'authorises_threshold_change':False,
    'authorises_model_retraining':False,
    'next_authorised_step':'independent semantic validation / supervised shadow evidence only' if verdict.startswith('PASS') else 'diagnose without post-hoc threshold or grouping changes',
    'limitation':'The complete Pilot 09 population has no authoritative semantic truth labels in this run. Coverage and contradiction are measured; accuracy is not established.',
}
atomic_json(OUT/'notebook69_verdict.json',verdict_record)

lines=[
    'NOTEBOOK 69 — FROZEN 59R5_MERGED SEMANTIC-CONTEXT TEST','',
    f'Verdict: {verdict}',f'Components: {n}',f'Pages: {paired.page.nunique()}',
    f'Notebook 67 RAW confident coverage: {raw_coverage:.4%} ({raw_selected}/{n})',
    f'Notebook 69 MERGED confident coverage: {merged_coverage:.4%} ({merged_selected}/{n})',
    f'Absolute coverage gain: {absolute_gain:.4%}',f'Relative coverage gain: {relative_gain:.4%}',
    f'RAW UNCERTAIN resolved: {int(paired.raw_uncertain_resolved.sum())}',
    f'RAW confident lost: {int(paired.raw_confident_lost.sum())}',
    f'RAW↔MERGED ANNOTATION/PRINT contradictions: {contradictions}',
    f'MERGED ensemble ANNOTATION/PRINT splits: {member_splits}',
    f'Pages with coverage regression: {regressed_pages}/{len(page_summary)}','',
    'GATES',*[f'- {k}: {v}' for k,v in gates.items()],'',
    'INTERPRETATION',
    '- Only semantic context changed: RAW component -> frozen 59R5_MERGED.',
    '- Model, calibration, thresholds, population, structural safety and correction authority stayed fixed.',
    '- No page, mask or correction output was modified.',
    '- Coverage improvement is not semantic-accuracy proof because authoritative truth is unavailable for the full Pilot population.',
    '- No automatic correction is authorised.'
]
(OUT/'executive_summary.txt').write_text('\n'.join(lines))
print('\n'.join(lines))

## 14. Software identity and standard Final Upload Package

In [ ]:
environment={
    'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,
    'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,
    'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()
}
atomic_json(OUT/'software_environment.json',environment)

experiment_identity={
    'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook69-1.0',
    'development_shadow_only':True,'promotion_authority':False,
    'predeclared_representation':'59R5_MERGED','merge_gap':MERGE_GAP,'merge_dilation':MERGE_DILATION,
    'maximum_merge_members':MAX_MERGE_MEMBERS,'scales':SCALES,'channels':EXPECTED_CHANNELS,
    'classes':CLASSES,'ensemble_members':[{'fold':x['fold'],'checkpoint_sha256':x['checkpoint_sha256'],'temperature':x['temperature']} for x in ensemble],
    'policy_thresholds':summary['thresholds'],'materiality':summary['materiality'],
    'population_sha256':sha256_file(INPUTS/'exact_notebook67_pilot_population.csv'),
    'merged_scores_sha256':sha256_file(TABLES/'merged_semantic_scores.csv'),
    'paired_results_sha256':sha256_file(TABLES/'raw_vs_merged_paired_components.csv'),
    'source_identity':source_identity,'software':environment,
}
atomic_json(OUT/'experiment_identity.json',experiment_identity)

upload_items=[
    OUT/'executive_summary.txt',OUT/'notebook69_summary.json',OUT/'notebook69_verdict.json',OUT/'experiment_identity.json',OUT/'software_environment.json',
    TABLES/'merged_representation_manifest.csv',TABLES/'merged_semantic_scores.csv',TABLES/'raw_vs_merged_paired_components.csv',
    TABLES/'comparison_by_page.csv',TABLES/'raw_to_merged_transition_matrix.csv',TABLES/'merged_semantic_by_frozen_structural_safety.csv',
    TABLES/'notebook69_two_axis_shadow_matrix.csv',REVIEW/'visual_audit_queue.csv',
    AUDIT/'source_identity.json',AUDIT/'read_only_and_authority_audit.json',AUDIT/'timing_by_page.csv',
    AUDIT/'observed_page_identity.csv',AUDIT/'unchanged_correction_authority_and_structural_safety.csv',
]
for p in upload_items: require(p.exists(),f'Upload artefact missing: {p}')
manifest=[]
for p in upload_items:
    destination=UPLOAD/p.parent.name/p.name
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(p,destination)
    manifest.append({'source':str(p),'relative_path':str(destination.relative_to(UPLOAD)),'bytes':destination.stat().st_size,'sha256':sha256_file(destination)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['relative_path'] for x in manifest))
(UPLOAD/'README.txt').write_text('Notebook 69 review package. Start with executive_summary.txt and notebook69_verdict.json. This package is shadow evidence only; it authorises no automatic correction.\n')

zip_path=OUT/'notebook69_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file(): z.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Final upload ZIP creation failed.')
print('Final upload package:',zip_path)
print('Notebook 69 complete:',OUT)

## Outputs to inspect first

1. `executive_summary.txt`
2. `notebook69_verdict.json`
3. `notebook69_summary.json`
4. `tables/raw_to_merged_transition_matrix.csv`
5. `tables/comparison_by_page.csv`
6. `tables/raw_vs_merged_paired_components.csv`
7. `review/visual_audit_queue.csv`
8. `tables/merged_semantic_by_frozen_structural_safety.csv`
9. `audit/read_only_and_authority_audit.json`
10. `notebook69_upload_bundle.zip`

A coverage increase is useful only if the frozen high-risk contradiction controls remain acceptable. Notebook 69 does not contain full authoritative semantic truth for the Pilot population, so it cannot by itself establish production accuracy or authorise automatic correction.